## STAGE 4 — Cross-Validation in Independent GEO Cohorts

Checks whether the Stage 2 DEGs change in the SAME direction in independent GEO series, and plots a discovery-vs-validation logFC concordance chart.

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import binomtest
import GEOparse
import os
import requests

Show the directories

In [14]:
# Paths
TABLES_DIR = r"C:\Users\naemi\Documents\Cis-platin\Cisplatin-Resistant-Ovarian-Cancer-Drug-Discovery\Results\Tables"
FIGS_DIR = r"C:\Users\naemi\Documents\Cis-platin\Cisplatin-Resistant-Ovarian-Cancer-Drug-Discovery\Results\Figures\plots"
CACHE_DIR = r"./geo_cache"
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(FIGS_DIR, exist_ok=True)
 

Download the data from NCBI GEO

In [15]:
# Reliable download function

def get_or_download_geo(geo_id, destdir=CACHE_DIR):
    local_file = os.path.join(destdir, f"{geo_id}_family.soft.gz")
    if not os.path.exists(local_file) or os.path.getsize(local_file) < 1024 * 100:
        stub = geo_id[:-3] + "nnn"
        url = f"https://ftp.ncbi.nlm.nih.gov/geo/series/{stub}/{geo_id}/soft/{geo_id}_family.soft.gz"
        print(f"[{geo_id}] Downloading archive from {url} ...")
        headers = {"User-Agent": "Mozilla/5.0"}
        r = requests.get(url, stream=True, headers=headers, timeout=120)
        r.raise_for_status()
        with open(local_file, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
        print(f"[{geo_id}] Download complete.")
    else:
        print(f"[{geo_id}] Using cached archive: {local_file}")
    return GEOparse.get_GEO(filepath=local_file, destdir=destdir)
 

# Validate Series

In [16]:
# validate_series()

def validate_series(geo_id, resistant_regex, sensitive_regex, candidate_genes):
    g = get_or_download_geo(geo_id)
 
    # Identify samples from titles
    sample_titles = pd.Series({s: g.gsms[s].metadata.get("title", [""])[0] for s in g.gsms})
    is_res = sample_titles.str.contains(resistant_regex, regex=True, case=False)
    is_sen = sample_titles.str.contains(sensitive_regex, regex=True, case=False)
 
    res_samples = sample_titles[is_res].index.tolist()
    sen_samples = sample_titles[is_sen].index.tolist()
 
    print(f"[{geo_id}] Matched {len(res_samples)} resistant and {len(sen_samples)} sensitive samples.")
    if len(res_samples) < 2 or len(sen_samples) < 2:
        print(f"[{geo_id}] WARNING: Insufficient samples matched. Titles found:\n{sample_titles}")
        return pd.DataFrame()
 
    # Build expression matrix
    frames = [g.gsms[s].table.set_index("ID_REF")["VALUE"].rename(s) for s in g.gsms]
    e = pd.concat(frames, axis=1).apply(pd.to_numeric, errors="coerce")
 
    max_val = e.max().max()
    if max_val > 30:
        print(f"[{geo_id}] Data appears non-log-scaled (max={max_val:.1f}), applying log2(x+1)")
        e = np.log2(e.clip(lower=0) + 1)
    else:
        print(f"[{geo_id}] Data already looks log2-scaled (max={max_val:.1f})")
 
    # Map probe ID -> Gene Symbol using the GPL platform table
    gpl = list(g.gpls.values())[0]
    annot = gpl.table
 
    sym_col = None
    for candidate in ["Symbol", "Gene Symbol", "gene_assignment", "GENE_SYMBOL", "GENE", "gene_symbol"]:
        if candidate in annot.columns:
            sym_col = candidate
            break
 
    if sym_col is not None:
        mapping = annot.set_index("ID")[sym_col].dropna().astype(str)
        mapping = mapping.apply(lambda x: x.split("//")[0].split(";")[0].strip().upper())
        e["Gene"] = e.index.map(mapping)
        e = e.dropna(subset=["Gene"]).groupby("Gene").mean()
        print(f"[{geo_id}] Successfully mapped probes to {e.shape[0]} unique gene symbols.")
    else:
        print(f"[{geo_id}] WARNING: No gene symbol column found in platform table — "
              f"results below may be empty or unreliable.")
 
    # Intersect with candidate genes (case-insensitive match, matching the
    
    clean_candidates = [str(gene).strip().upper() for gene in candidate_genes]
    genes_to_test = [gene for gene in clean_candidates if gene in e.index]
    print(f"[{geo_id}] Overlapping candidate genes found in dataset: {len(genes_to_test)} / {len(clean_candidates)}")
 
    out = []
    for gene in genes_to_test:
        r = e.loc[gene, res_samples].dropna().values
        s = e.loc[gene, sen_samples].dropna().values
        if len(r) < 2 or len(s) < 2:
            continue
        _, p = stats.ttest_ind(r, s, equal_var=False)
        out.append({
            "gene": gene,
            "logFC": float(np.mean(r) - np.mean(s)),
            "pval": float(p),
            "res_mean": float(np.mean(r)),
            "sen_mean": float(np.mean(s)),
            "cohort": geo_id,
        })
 
    return pd.DataFrame(out)
 
 

# Load discovery DEGs found from the earlier steps

In [17]:
# Load discovery DEGs, ensure gene symbols are uppercase for matching

sig_degs = pd.read_csv(os.path.join(TABLES_DIR, "11_DEGs_resistant_vs_sensitive.csv"))
sig_degs["gene"] = sig_degs["gene"].str.strip().str.upper()   
 
# Run validation across cohorts

val_58470 = validate_series(
    geo_id="GSE58470",
    resistant_regex=r"Pt1",                    
    sensitive_regex=r"IGROV-1_replicate",
    candidate_genes=sig_degs["gene"],
)
 
val_15372 = validate_series(
    geo_id="GSE15372",
    resistant_regex=r"Round5|resistant",
    sensitive_regex=r"parental",
    candidate_genes=sig_degs["gene"],
)
 
print(f"\nGSE58470 validated rows: {len(val_58470)}")
print(f"GSE15372 validated rows: {len(val_15372)}")
 
# Sanity check: confirm the scale fix actually took effect
for name, df in [("GSE58470", val_58470), ("GSE15372", val_15372)]:
    if not df.empty:
        print(f"[{name}] logFC range after fix: {df['logFC'].min():.3f} to {df['logFC'].max():.3f} "
              f"(should be roughly -10 to +10, NOT thousands)")
 
all_validation = pd.concat([val_58470, val_15372], ignore_index=True)
all_validation.to_csv(os.path.join(TABLES_DIR, "14_validation_results_by_cohort.csv"), index=False)
print(f"Saved {len(all_validation)} validated gene records.")


20-Sep-2026 02:12:36 INFO GEOparse - Parsing ./geo_cache\GSE58470_family.soft.gz: 
20-Sep-2026 02:12:36 DEBUG GEOparse - DATABASE: GeoMiame
20-Sep-2026 02:12:36 DEBUG GEOparse - SERIES: GSE58470
20-Sep-2026 02:12:36 DEBUG GEOparse - PLATFORM: GPL6947


[GSE58470] Using cached archive: ./geo_cache\GSE58470_family.soft.gz


20-Sep-2026 02:12:37 DEBUG GEOparse - SAMPLE: GSM1412114
20-Sep-2026 02:12:37 DEBUG GEOparse - SAMPLE: GSM1412115
20-Sep-2026 02:12:38 DEBUG GEOparse - SAMPLE: GSM1412116
20-Sep-2026 02:12:38 DEBUG GEOparse - SAMPLE: GSM1412117
20-Sep-2026 02:12:38 DEBUG GEOparse - SAMPLE: GSM1412118
20-Sep-2026 02:12:38 DEBUG GEOparse - SAMPLE: GSM1412119
20-Sep-2026 02:12:38 DEBUG GEOparse - SAMPLE: GSM1412120
20-Sep-2026 02:12:38 DEBUG GEOparse - SAMPLE: GSM1412121
20-Sep-2026 02:12:38 DEBUG GEOparse - SAMPLE: GSM1412122


[GSE58470] Matched 3 resistant and 3 sensitive samples.
[GSE58470] Data appears non-log-scaled (max=71072.4), applying log2(x+1)
[GSE58470] Successfully mapped probes to 25158 unique gene symbols.
[GSE58470] Overlapping candidate genes found in dataset: 212 / 248


20-Sep-2026 02:12:39 INFO GEOparse - Parsing ./geo_cache\GSE15372_family.soft.gz: 
20-Sep-2026 02:12:39 DEBUG GEOparse - DATABASE: GeoMiame
20-Sep-2026 02:12:39 DEBUG GEOparse - SERIES: GSE15372
20-Sep-2026 02:12:39 DEBUG GEOparse - PLATFORM: GPL570


[GSE15372] Using cached archive: ./geo_cache\GSE15372_family.soft.gz


C:\Users\naemi\AppData\Roaming\Python\Python314\site-packages\GEOparse\GEOparse.py:401: DtypeWarning: Columns (0: SPOT_ID) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")
20-Sep-2026 02:12:41 DEBUG GEOparse - SAMPLE: GSM385721
20-Sep-2026 02:12:41 DEBUG GEOparse - SAMPLE: GSM385722
20-Sep-2026 02:12:41 DEBUG GEOparse - SAMPLE: GSM385723
20-Sep-2026 02:12:41 DEBUG GEOparse - SAMPLE: GSM385724
20-Sep-2026 02:12:41 DEBUG GEOparse - SAMPLE: GSM385725
20-Sep-2026 02:12:41 DEBUG GEOparse - SAMPLE: GSM385726
20-Sep-2026 02:12:41 DEBUG GEOparse - SAMPLE: GSM385727
20-Sep-2026 02:12:41 DEBUG GEOparse - SAMPLE: GSM385728
20-Sep-2026 02:12:41 DEBUG GEOparse - SAMPLE: GSM385729
20-Sep-2026 02:12:42 DEBUG GEOparse - SAMPLE: GSM385730


[GSE15372] Matched 5 resistant and 5 sensitive samples.
[GSE15372] Data appears non-log-scaled (max=506237.2), applying log2(x+1)
[GSE15372] Successfully mapped probes to 22878 unique gene symbols.
[GSE15372] Overlapping candidate genes found in dataset: 239 / 248

GSE58470 validated rows: 212
GSE15372 validated rows: 239
[GSE58470] logFC range after fix: -5.582 to 3.708 (should be roughly -10 to +10, NOT thousands)
[GSE15372] logFC range after fix: -3.034 to 5.626 (should be roughly -10 to +10, NOT thousands)
Saved 451 validated gene records.


# Concordance

In [18]:
# Concordance: does direction match the discovery set

discovery_fc = sig_degs.set_index("gene")["logFC"]
concordance = all_validation.copy()
concordance["discovery_logFC"] = concordance["gene"].map(discovery_fc)
concordance = concordance.dropna(subset=["discovery_logFC"]) 
concordance["concordant"] = np.sign(concordance["logFC"]) == np.sign(concordance["discovery_logFC"])
concordance.to_csv(os.path.join(TABLES_DIR, "15_validation_results_by_cohort_concordance.csv"), index=False)    
print(f"Saved {len(concordance)} concordance records.")

Saved 451 concordance records.


In [19]:
# Binomial test: is concordance actually above 50% chance

print("\n--- Significance vs. chance (50%) ---")
for cohort, sub in concordance.groupby("cohort"):
    n_conc = int(sub["concordant"].sum())
    n_total = len(sub)
    pct = n_conc / n_total
    result = binomtest(n_conc, n_total, p=0.5, alternative="two-sided")
    direction = "ABOVE chance (real signal)" if (pct > 0.5 and result.pvalue < 0.05) else \
                "BELOW chance (check labels/regex!)" if (pct < 0.5 and result.pvalue < 0.05) else \
                "NOT distinguishable from chance"
    print(f"{cohort}: {n_conc}/{n_total} = {pct:.1%} concordant, "
          f"p={result.pvalue:.4f} vs 50% → {direction}")
 


--- Significance vs. chance (50%) ---
GSE15372: 107/239 = 44.8% concordant, p=0.1204 vs 50% → NOT distinguishable from chance
GSE58470: 129/212 = 60.8% concordant, p=0.0019 vs 50% → ABOVE chance (real signal)


# Find replicated genes

In [20]:
# Replicated genes (concordant in >=2 cohorts, relax to >=1 if too few)

concordant_counts = concordance.groupby("gene")["concordant"].sum()
replicated_genes = concordant_counts[concordant_counts >= 2].index.tolist()
 
if len(replicated_genes) < 5:
    print("\nFew genes replicated in >=2 cohorts — relaxing to >=1 cohort")
    replicated_genes = concordant_counts[concordant_counts >= 1].index.tolist()
 
pd.DataFrame({"gene": replicated_genes}).to_csv(
    os.path.join(TABLES_DIR, "15_replicated_genes.csv"), index=False
)
print(f"\nReplicated genes: {len(replicated_genes)} / {len(sig_degs)} discovery DEGs")
 


Replicated genes: 61 / 248 discovery DEGs


# Make plots

In [ ]:
# discovery vs validation logFC concordance scatter Plots 

fig, axes = plt.subplots(1, len(concordance["cohort"].unique()),
                          figsize=(6 * len(concordance["cohort"].unique()), 5), squeeze=False)
for ax, (cohort, sub) in zip(axes[0], concordance.groupby("cohort")):
    colors = np.where(sub["concordant"], "seagreen", "lightcoral")
    ax.scatter(sub["discovery_logFC"], sub["logFC"], c=colors, alpha=0.7, s=25)
    lims = [min(sub["discovery_logFC"].min(), sub["logFC"].min()) - 0.5,
            max(sub["discovery_logFC"].max(), sub["logFC"].max()) + 0.5]
    ax.plot(lims, lims, "k--", linewidth=0.8)
    ax.axhline(0, color="gray", linewidth=0.5)
    ax.axvline(0, color="gray", linewidth=0.5)
    ax.set_xlabel("Discovery logFC (GSE47856)")
    ax.set_ylabel(f"Validation logFC ({cohort})")
    n_concordant = sub["concordant"].sum()
    ax.set_title(f"{cohort}: {n_concordant}/{len(sub)} concordant")
 
plt.tight_layout()
plt.savefig(os.path.join (FIGS_DIR, "09_stage4_discovery_vs_validation_logFC.png"), dpi=1000)
plt.close()

In [ ]:
# Bar chart of replication count — FILTERED for readability

plot_counts = concordant_counts[concordant_counts >= 1].sort_values()   # drop the 0-count genes
if len(plot_counts) > 60:
    print(f"NOTE: {len(plot_counts)} genes with >=1 concordant cohort — showing top 60 by count")
    plot_counts = plot_counts.tail(60)
 
plt.figure(figsize=(8, max(4, len(plot_counts) * 0.18)))
colors_bar = np.where(plot_counts >= 2, "seagreen", "lightgray")
plt.barh(plot_counts.index.astype(str), plot_counts.values, color=colors_bar)
plt.xlabel("Number of cohorts with concordant direction")
plt.title("Replication count per candidate gene\n(genes with 0 concordant cohorts omitted)")
plt.tight_layout()
plt.savefig(os.path.join(FIGS_DIR, "10_stage4_replication_count_bar.png"), dpi=1000)
plt.close()
 
print(f"\nPlots saved to {FIGS_DIR}")
print("NEXT: check the binomial test results above BEFORE proceeding to Stage 5.")

NOTE: 175 genes with >=1 concordant cohort — showing top 60 by count

Plots saved to C:\Users\naemi\Documents\Cis-platin\Cisplatin-Resistant-Ovarian-Cancer-Drug-Discovery\Results\Figures\plots
NEXT: check the binomial test results above BEFORE proceeding to Stage 5.
